# OTTO — T4×2 Multi‑GPU Co‑Vis + Dask XGBRanker + Radek CV (cuDF & dask‑cuDF)
**Updated:** 2025-10-27

This notebook is a **dual‑GPU (T4×2)** version:
- Multi‑GPU co‑visitation matrices computed with **dask‑cuDF** over **dask‑cuda** cluster.
- **Group-wise ranking** with **`xgboost.dask.DaskXGBRanker`** (each session = one group).
- **Radek split (last 24h)** for local CV; compare **RULE / RANK / BLEND** curves.
- Safe **RMM pool** to reduce OOM on T4 16GB x 2.

## 0. Environment & GPU Pooling

In [ ]:
# Ensure RAPIDS 24.02+ on Kaggle T4 works; set up a two‑GPU Dask cluster
import os, warnings
warnings.filterwarnings("ignore")

# Use both GPUs 0,1
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# RMM pooled allocator for cuDF/cuPy (reduce fragmentation)
import rmm, cupy as cp, cudf

# Initialize ~12GB per-GPU pool (tweak if OOM)
POOL_GB = int(os.environ.get("RMM_POOL_GB", "12"))
rmm.reinitialize(pool_allocator=True, initial_pool_size=POOL_GB * 1024**3)
cudf.set_allocator(rmm.rmm_cupy_allocator)
cp.cuda.set_allocator(rmm.rmm_cupy_allocator)

print("RMM pool initialized:", POOL_GB, "GB per process")

# Dask‑CUDA cluster across both GPUs
from dask_cuda import LocalCUDACluster
from dask.distributed import Client

cluster = LocalCUDACluster(
    CUDA_VISIBLE_DEVICES=os.environ["CUDA_VISIBLE_DEVICES"],
    rmm_pool_size=f"{POOL_GB}GB",
    protocol="tcp",
)
client = Client(cluster)
client

## 1. Imports & Constants

In [ ]:
import glob, gc, itertools, numpy as np, pandas as pd
from collections import Counter, defaultdict

import dask_cudf as dc
import cudf

VER = 5
READ_CT = 5   # keep small batches
# We'll partition by aid_x ranges similar in spirit to DISK_PIECES logic
DISK_PIECES = 4
SIZE = 1.86e6 / DISK_PIECES

type_labels = {'clicks':0, 'carts':1, 'orders':2}
TYPE_W = {0:1, 1:6, 2:3}

print("DISK_PIECES:", DISK_PIECES)

## 2. Source Parquet Discovery

In [ ]:
# Use Columbia2131 parquet dataset (train & test)
train_files = sorted(glob.glob('../input/otto-chunk-data-inparquet-format/*_parquet/*'))
test_files  = sorted(glob.glob('../input/otto-chunk-data-inparquet-format/test_parquet/*'))
len(train_files), len(test_files)

## 3. Multi‑GPU Co‑Visitation Matrices with dask‑cuDF

In [ ]:
# Utility: read & normalize a parquet path into cuDF with int ts (seconds) and mapped type
def _read_norm_pq(path):
    gdf = cudf.read_parquet(path, columns=["session","aid","ts","type"])
    if gdf["ts"].max() > 1e10:
        gdf["ts"] = (gdf["ts"] / 1000).astype("int32")
    else:
        gdf["ts"] = gdf["ts"].astype("int32")
    if gdf["type"].dtype.kind in ("U","S","O"):
        gdf["type"] = gdf["type"].map(type_labels).astype("int8")
    return gdf

# Build a dask‑cuDF from the normalized partitions
def read_dc(paths, npartitions=24):
    # map_paths to delayed cudf frames
    dfs = [dc.from_cudf(_read_norm_pq(p), npartitions=1) for p in paths]
    ddf = dc.concat(dfs, interleave_partitions=True)
    return ddf.repartition(npartitions=npartitions)

# Pairing within session with time window & top‑k per key
def covis_matrix(ddf, window_sec, filter_types=None, topk=20, part_id=0, part_total=1):
    g = ddf
    if filter_types is not None:
        g = g[g["type"].isin(filter_types)]

    # Keep last 30 per session
    g = g.map_partitions(lambda df: df.sort_values(["session","ts"], ascending=[True, False]))
    g = g.assign(n=g.groupby("session").cumcount())
    g = g[g["n"] < 30].drop("n", axis=1)

    # Split on aid_x ranges to control memory (like DISK_PIECES)
    # We must self-join by session; we'll do partition-wise then filter by aid_x range
    left  = g.rename(columns={"aid":"aid_x","ts":"ts_x","type":"type_x"})[["session","aid_x","ts_x","type_x"]]
    right = g.rename(columns={"aid":"aid_y","ts":"ts_y","type":"type_y"})[["session","aid_y","ts_y","type_y"]]

    pairs = left.merge(right, on="session", how="inner")
    pairs = pairs[(pairs["aid_x"] != pairs["aid_y"]) & ((pairs["ts_x"] - pairs["ts_y"]).abs() < window_sec)]

    # range filter for aid_x
    low  = part_id * SIZE
    high = (part_id + 1) * SIZE
    pairs = pairs[(pairs["aid_x"] >= low) & (pairs["aid_x"] < high)]

    return pairs

def agg_weight_and_top(df_pairs, weight_kind="type", topk=20):
    df = df_pairs
    if weight_kind == "type":
        # clicks:1, carts:6, orders:3
        w = cudf.Series(df["type_y"].map(cudf.Series(TYPE_W)))
    elif weight_kind == "const":
        w = cudf.Series(cp.ones(len(df), dtype=cp.float32))
    elif weight_kind == "time":
        # linear time weighting
        ts0, ts1 = 1659304800, 1662328791
        w = 1 + 3 * (df["ts_x"] - ts0) / (ts1 - ts0)
    else:
        raise ValueError("Unknown weight_kind")

    df = cudf.DataFrame({"aid_x": df["aid_x"], "aid_y": df["aid_y"], "wgt": w.astype("float32")})
    df = df.drop_duplicates(subset=["session","aid_x","aid_y"], keep="first", ignore_index=True) if "session" in df_pairs else df
    grp = df.groupby(["aid_x","aid_y"]).agg({"wgt":"sum"}).reset_index()

    # Top‑k per aid_x by weight
    grp = grp.sort_values(["aid_x","wgt"], ascending=[True, False])
    grp["n"] = grp.groupby("aid_x").cumcount()
    grp = grp[grp["n"] < topk].drop("n", axis=1)
    return grp

# Compute and save all three matrices across parts on two GPUs
def compute_all_covis(train_paths, disk_pieces=DISK_PIECES):
    # dask read once (still using cudf per-file internally)
    ddf = read_dc(train_paths, npartitions=64)
    ddf = ddf.persist()

    for part in range(disk_pieces):
        print(f"\n### PART {part+1}/{disk_pieces} — Carts/Orders (type‑weighted)")
        p = covis_matrix(ddf, window_sec=24*3600, filter_types=None, topk=15, part_id=part, part_total=disk_pieces)
        pq = p.map_partitions(agg_weight_and_top, weight_kind="type", topk=15).persist()
        df = pq.compute()
        df.to_parquet(f"top_15_carts_orders_v{VER}_{part}.pqt")

    # buy2buy
    print(f"\n### PART 1/1 — Buy2Buy (const weight)")
    p = covis_matrix(ddf[ddf["type"].isin([1,2])], window_sec=14*24*3600, filter_types=None, topk=15, part_id=0, part_total=1)
    pq = p.map_partitions(agg_weight_and_top, weight_kind="const", topk=15).persist()
    df = pq.compute()
    df.to_parquet(f"top_15_buy2buy_v{VER}_0.pqt")

    # clicks (time‑weighted)
    for part in range(disk_pieces):
        print(f"\n### PART {part+1}/{disk_pieces} — Clicks (time‑weighted)")
        p = covis_matrix(ddf, window_sec=24*3600, filter_types=None, topk=20, part_id=part, part_total=disk_pieces)
        pq = p.map_partitions(agg_weight_and_top, weight_kind="time", topk=20).persist()
        df = pq.compute()
        df.to_parquet(f"top_20_clicks_v{VER}_{part}.pqt")

    del ddf
    gc.collect()

compute_all_covis(train_files, disk_pieces=DISK_PIECES)

## 4. Load Test & Co‑Vis Dictionaries

In [ ]:
# Test data as pandas (sufficient for downstream per‑session loops)
def load_test():
    dfs = []
    for p in test_files:
        df = pd.read_parquet(p, columns=["session","aid","ts","type"])
        df["ts"] = (df["ts"]/1000).astype("int32") if df["ts"].max()>1e10 else df["ts"].astype("int32")
        if df["type"].dtype == "O":
            df["type"] = df["type"].map(type_labels).astype("int8")
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

test_df = load_test()
print("Test:", test_df.shape)

def pqt_to_weighted_and_rank(paths):
    from collections import defaultdict
    weights, ranks = defaultdict(dict), defaultdict(dict)
    for p in paths:
        df = pd.read_parquet(p, columns=["aid_x","aid_y","wgt"])
        df = df.sort_values(["aid_x","wgt"], ascending=[True, False])
        df["rank"] = df.groupby("aid_x").cumcount().astype("int32")
        for ax, ay, w, rk in zip(df["aid_x"], df["aid_y"], df["wgt"], df["rank"]):
            weights[int(ax)][int(ay)] = float(w)
            ranks[int(ax)][int(ay)]   = int(rk)
    return dict(weights), dict(ranks)

CLICK_PQTS = sorted(glob.glob(f"top_20_clicks_v{VER}_*.pqt"))
BUYS_PQTS  = sorted(glob.glob(f"top_15_carts_orders_v{VER}_*.pqt"))
B2B_PQTS   = sorted(glob.glob(f"top_15_buy2buy_v{VER}_*.pqt"))

click_w, click_r = pqt_to_weighted_and_rank(CLICK_PQTS)
buys_w,  buys_r  = pqt_to_weighted_and_rank(BUYS_PQTS)
b2b_w,   b2b_r   = pqt_to_weighted_and_rank(B2B_PQTS)

# Top clicks/orders from test (types are numeric already)
top_clicks = test_df.loc[test_df["type"]==0, "aid"].value_counts().index.values[:20]
top_orders = test_df.loc[test_df["type"]==2, "aid"].value_counts().index.values[:20]
top_clicks_set, top_orders_set = set(top_clicks), set(top_orders)
print("Loaded co‑vis dicts. Click items:", len(click_w), "Buy2Buy items:", len(b2b_w))

## 5. Rule Suggestors (fixed)

In [ ]:
def suggest_clicks(df):
    aids = df.aid.tolist()
    types = df.type.tolist()
    unique_aids = list(dict.fromkeys(aids[::-1]))
    if len(unique_aids) >= 20:
        weights = np.logspace(0.1,1,len(aids),base=2, endpoint=True) - 1
        sc = Counter()
        for aid,w,t in zip(aids,weights,types):
            sc[aid] += float(w) * TYPE_W[int(t)]
        return [k for k,_ in sc.most_common(20)]
    # use clicks co-vis
    aids2 = list(itertools.chain(*[click_w.get(a, {}).keys() for a in unique_aids]))
    top_aids2 = [aid2 for aid2,_ in Counter(aids2).most_common(20) if aid2 not in unique_aids]
    res = unique_aids + top_aids2[:20 - len(unique_aids)]
    return res + list(top_clicks)[:20 - len(res)]

def suggest_buys(df):
    aids = df.aid.tolist()
    types = df.type.tolist()
    unique_aids = list(dict.fromkeys(aids[::-1]))
    dfb = df[df["type"].isin([1,2])]
    unique_buys = list(dict.fromkeys(dfb.aid.tolist()[::-1]))
    if len(unique_aids) >= 20:
        weights = np.logspace(0.5,1,len(aids),base=2, endpoint=True) - 1
        sc = Counter()
        for aid,w,t in zip(aids,weights,types):
            sc[aid] += float(w) * TYPE_W[int(t)]
        aids3 = list(itertools.chain(*[b2b_w.get(a, {}).keys() for a in unique_buys]))
        for a in aids3: sc[a] += 0.1
        return [k for k,_ in sc.most_common(20)]
    aids2 = list(itertools.chain(*[buys_w.get(a, {}).keys() for a in unique_aids]))
    aids3 = list(itertools.chain(*[b2b_w.get(a,  {}).keys() for a in unique_buys]))
    top_aids2 = [aid2 for aid2,_ in Counter(aids2 + aids3).most_common(20) if aid2 not in unique_aids]
    res = unique_aids + top_aids2[:20 - len(unique_aids)]
    return res + list(top_orders)[:20 - len(res)]

## 6. Feature Building (minimal, per session)

In [ ]:
def build_session_seeds(df_hist):
    aids  = df_hist.aid.tolist()
    types = df_hist.type.tolist()
    unique_aids = list(dict.fromkeys(aids[::-1]))
    unique_buys = list(dict.fromkeys(df_hist[df_hist["type"].isin([1,2])].aid.tolist()[::-1]))
    return aids, types, unique_aids, unique_buys

def recent_type_score(aids, types, lo=0.1, hi=1.0):
    w = np.logspace(lo, hi, len(aids), base=2, endpoint=True) - 1.0
    sc = Counter()
    for aid, ww, t in zip(aids, w, types):
        sc[aid] += float(ww) * TYPE_W[int(t)]
    return sc

def aggregate_neighbors(unique_aids, unique_buys):
    agg_click, agg_buys, agg_b2b = Counter(), Counter(), Counter()
    for a in unique_aids:
        for b,w in click_w.get(a, {}).items(): agg_click[b] += w
        for b,w in buys_w.get(a,  {}).items(): agg_buys[b]  += w
    for a in unique_buys:
        for b,w in b2b_w.get(a,   {}).items(): agg_b2b[b]   += w
    return agg_click, agg_buys, agg_b2b

def build_candidates(df_hist, k_click=120, k_buys=120):
    aids, types, unique_aids, unique_buys = build_session_seeds(df_hist)
    agg_click, agg_buys, agg_b2b = aggregate_neighbors(unique_aids, unique_buys)
    cand_clicks = set(unique_aids) | set([b for b,_ in agg_click.most_common(k_click)])
    cand_buys   = set(unique_aids) | set([b for b,_ in (agg_buys+agg_b2b).most_common(k_buys)])
    return (aids, types, unique_aids, unique_buys, agg_click, agg_buys, agg_b2b, cand_clicks, cand_buys)

def best_rank_over_seeds(c, seeds, ranks, default=9999):
    best = default
    for s in seeds:
        r = ranks.get(s, {}).get(c, default)
        if r < best: best = r
    return best

FEATURE_COLS = [
    'in_hist','rec_rule',
    'w_main','best_main',
    'w_aux1','best_aux1',
    'best_b2b',
    'sess_len','gap',
    'is_top_click','is_top_order'
]

def make_features_for_head(df_hist, head, candidates,
                           aids, types, unique_aids, unique_buys,
                           agg_click, agg_buys, agg_b2b):
    rec_sc  = recent_type_score(aids, types)
    sesslen = len(aids)
    last_ts = int(df_hist.ts.max()) if sesslen else 0
    rows, keys = [], []
    for c in candidates:
        in_hist = 1 if c in rec_sc else 0
        w_click = float(agg_click.get(c, 0.0))
        w_buys  = float(agg_buys.get(c, 0.0))
        w_b2b   = float(agg_b2b.get(c, 0.0))  # corrected
        r_click = best_rank_over_seeds(c, unique_aids, click_r)
        r_buys  = best_rank_over_seeds(c, unique_aids, buys_r)
        r_b2b   = best_rank_over_seeds(c, unique_buys, b2b_r)
        f_best_click = 1.0/(1.0 + r_click)
        f_best_buys  = 1.0/(1.0 + r_buys)
        f_best_b2b   = 1.0/(1.0 + r_b2b)
        f_rec_sc = float(rec_sc.get(c, 0.0))
        if c in df_hist.aid.values:
            gap = last_ts - int(df_hist[df_hist.aid==c].ts.max())
        else:
            gap = 10*24*3600
        is_top_click = int(c in top_clicks_set)
        is_top_order = int(c in top_orders_set)
        if head=='clicks':
            rows.append([in_hist, f_rec_sc, w_click, f_best_click,
                         w_buys, f_best_buys, f_best_b2b,
                         sesslen, gap, is_top_click, is_top_order])
        else:
            rows.append([in_hist, f_rec_sc, w_buys, f_best_buys,
                         w_click, f_best_click, f_best_b2b,
                         sesslen, gap, is_top_click, is_top_order])
        keys.append(c)
    X = pd.DataFrame(rows, columns=FEATURE_COLS)
    return X, keys

## 7. Training with **DaskXGBRanker** (group = session)

In [ ]:
# We'll create a lightweight training set via Radek split per train parquet
import random
from tqdm import tqdm

def build_train_rows_from_session(df_sess):
    if df_sess.empty: return [], []
    split_ts = int(df_sess.ts.max()) - 24*60*60
    hist = df_sess[df_sess.ts <  split_ts]
    fut  = df_sess[df_sess.ts >= split_ts]
    if hist.empty or fut.empty: return [], []
    (aids, types, unique_aids, unique_buys,
     agg_click, agg_buys, agg_b2b,
     cand_clicks, cand_buys) = build_candidates(hist)
    labs_clicks = set(fut.loc[fut['type']==0,'aid'].tolist())
    labs_buys   = set(fut.loc[fut['type'].isin([1,2]),'aid'].tolist())
    Xc, kc = make_features_for_head(hist, 'clicks', cand_clicks,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    yc = [1 if a in labs_clicks else 0 for a in kc]
    Xb, kb = make_features_for_head(hist, 'buys', cand_buys,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    yb = [1 if a in labs_buys else 0 for a in kb]
    return [(Xc, yc)], [(Xb, yb)]

def build_training_dataset(paths, max_sessions=60000, seed=42):
    random.seed(seed)
    files = paths.copy()
    random.shuffle(files)
    Xc_list, yc_list, Gc_list = [], [], []
    Xb_list, yb_list, Gb_list = [], [], []
    seen = 0
    for f in tqdm(files, total=len(files), desc="Scan train files"):
        df = pd.read_parquet(f, columns=["session","aid","ts","type"])
        df["ts"] = (df["ts"]/1000).astype("int32") if df["ts"].max()>1e10 else df["ts"].astype("int32")
        if df["type"].dtype=="O":
            df["type"] = df["type"].map(type_labels).astype("int8")
        for sid, g in df.groupby("session"):
            xc, xb = build_train_rows_from_session(g)
            for Xc,yc in xc:
                Xc_list.append(Xc)
                yc_list += yc
                Gc_list.append(len(yc))
            for Xb,yb in xb:
                Xb_list.append(Xb)
                yb_list += yb
                Gb_list.append(len(yb))
            seen += 1
            if seen >= max_sessions: break
        if seen >= max_sessions: break
    Xc = pd.concat(Xc_list, ignore_index=True) if Xc_list else pd.DataFrame()
    Xb = pd.concat(Xb_list, ignore_index=True) if Xb_list else pd.DataFrame()
    return (Xc, np.array(yc_list, np.int8), np.array(Gc_list, np.int32),
            Xb, np.array(yb_list, np.int8), np.array(Gb_list, np.int32))

# Build small-ish dataset to fit quickly on Kaggle T4×2
Xc, yc, Gc, Xb, yb, Gb = build_training_dataset(train_files, max_sessions=80000)
print("Clicks:", Xc.shape, yc.mean() if len(yc) else None, "groups:", len(Gc))
print("Buys  :", Xb.shape, yb.mean() if len(yb) else None, "groups:", len(Gb))

# DaskXGBRanker
from xgboost.dask import DaskXGBRanker

def fit_ranker(X_df, y, group_sizes, label="clicks"):
    if X_df.empty or len(np.unique(y))<2:
        print(f"[{label}] not enough data, skip.")
        return None
    import dask.array as da
    import dask.dataframe as dd
    # Move features to dask (CPU backed is fine; training happens on GPUs)
    dX = dd.from_pandas(X_df, npartitions=32)
    dy = dd.from_pandas(pd.Series(y), npartitions=32)
    dg = dd.from_pandas(pd.Series(group_sizes), npartitions=1)  # small
    ranker = DaskXGBRanker(
        n_estimators=300, max_depth=7, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9,
        reg_alpha=1e-3, reg_lambda=1.0,
        tree_method="gpu_hist", predictor="gpu_predictor",
        objective="rank:pairwise", random_state=2025,
        nthread=-1
    )
    ranker.client = client
    ranker.fit(dX, dy, group=dg)
    return ranker

ranker_clicks = fit_ranker(Xc, yc, Gc, "clicks")
ranker_buys   = fit_ranker(Xb, yb, Gb, "buys")

## 8. Rank & Blend Utilities

In [ ]:
def rank_and_take20_with_blend(model, X, keys, fallback_sorted, rule_score=None, alpha=None):
    if X is None or len(keys)==0:
        return fallback_sorted[:20]
    # model score
    if model is not None:
        try:
            preds = model.predict_proba(X)[:,1]  # DaskRanker does not expose predict_proba
        except Exception:
            try:
                preds = model.predict(X)
            except Exception:
                preds = np.zeros(len(keys), dtype=np.float32)
    else:
        preds = np.zeros(len(keys), dtype=np.float32)

    if alpha is None or rule_score is None:
        scores = preds
    else:
        scores = alpha * preds + (1.0 - alpha) * rule_score

    order = np.argsort(-scores)
    ranked = [keys[i] for i in order]
    out, used = [], set()
    for a in ranked + fallback_sorted:
        if a not in used:
            out.append(a); used.add(a)
        if len(out)==20: break
    return out

## 9. Radek CV（last 24h）— RULE / RANK / BLEND

In [ ]:
CUT_TS = int(test_df["ts"].max()) - 24*60*60

def _rule_only_predict(df_hist):
    return {'clicks': suggest_clicks(df_hist), 'buys': suggest_buys(df_hist)}

def _ranker_only_predict(df_hist):
    (aids, types, unique_aids, unique_buys,
     agg_click, agg_buys, agg_b2b,
     cand_clicks, cand_buys) = build_candidates(df_hist)
    Xc, kc = make_features_for_head(df_hist, 'clicks', cand_clicks,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    Xb, kb = make_features_for_head(df_hist, 'buys', cand_buys,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    fb_c = suggest_clicks(df_hist); fb_b = suggest_buys(df_hist)
    cc = rank_and_take20_with_blend(ranker_clicks, Xc.values if not Xc.empty else None, list(kc), fb_c, None, None)
    bb = rank_and_take20_with_blend(ranker_buys,   Xb.values if not Xb.empty else None, list(kb), fb_b, None, None)
    return {'clicks': cc, 'buys': bb}

def _blend_predict(df_hist, alpha=0.35):
    (aids, types, unique_aids, unique_buys,
     agg_click, agg_buys, agg_b2b,
     cand_clicks, cand_buys) = build_candidates(df_hist)
    Xc, kc = make_features_for_head(df_hist, 'clicks', cand_clicks,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    Xb, kb = make_features_for_head(df_hist, 'buys', cand_buys,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    fb_c = suggest_clicks(df_hist); fb_b = suggest_buys(df_hist)
    rule_c = (Xc['rec_rule'] + Xc['w_main'] + 0.2*Xc['best_main']).values if not Xc.empty else None
    rule_b = (Xb['rec_rule'] + Xb['w_main'] + 0.2*Xb['best_main']).values if not Xb.empty else None
    cc = rank_and_take20_with_blend(ranker_clicks, Xc.values if not Xc.empty else None, list(kc), fb_c, rule_c, alpha)
    bb = rank_and_take20_with_blend(ranker_buys,   Xb.values if not Xb.empty else None, list(kb), fb_b, rule_b, alpha)
    return {'clicks': cc, 'buys': bb}

df_all = test_df.copy()
df_all["ts"] = df_all["ts"].astype(np.int64)
has_hist = df_all.groupby("session")["ts"].min() < CUT_TS
has_fut  = df_all.groupby("session")["ts"].max() >= CUT_TS
valid_sessions = has_hist[has_hist & has_fut].index
print("Valid sessions for CV:", len(valid_sessions))

def _recall_at20(pred, truth):
    if not truth: return 0.0
    hit = len(set(pred) & truth)
    return hit / len(truth)

def _eval_mode(predict_fn, mode_name):
    rc_list, rb_list = [], []
    for sid, g in df_all[df_all["session"].isin(valid_sessions)].groupby("session"):
        hist = g[g["ts"] <  CUT_TS]
        fut  = g[g["ts"] >= CUT_TS]
        if hist.empty or fut.empty: continue
        pred = predict_fn(hist)
        gt_clicks = set(fut.loc[fut['type']==0, 'aid'].tolist())
        gt_buys   = set(fut.loc[fut['type'].isin([1,2]), 'aid'].tolist())
        rc_list.append(_recall_at20(pred['clicks'], gt_clicks))
        rb_list.append(_recall_at20(pred['buys'],   gt_buys))
    clicks_R = float(np.mean(rc_list)) if rc_list else 0.0
    buys_R   = float(np.mean(rb_list)) if rb_list else 0.0
    total = 0.10*clicks_R + 0.90*buys_R
    print(f"[{mode_name}] clicks_R={clicks_R:.5f}  buys_R={buys_R:.5f}  ==> score={total:.5f}")
    return clicks_R, buys_R, total

_eval_mode(_rule_only_predict,  "RULE-ONLY")
_eval_mode(_ranker_only_predict,"RANKER-ONLY")
_eval_mode(lambda h: _blend_predict(h, alpha=0.35), "BLEND-0.35")

## 10. Batch Inference & Submissions

In [ ]:
def _rank_take20_with_fallback(scores_keys_per_sess, fallback_per_sess):
    out = {}
    for sid, pairs in scores_keys_per_sess.items():
        pairs.sort(key=lambda x: -x[0])
        used, res = set(), []
        for sc, aid in pairs:
            if aid not in used:
                res.append(aid); used.add(aid)
            if len(res)==20: break
        if len(res)<20:
            for aid in fallback_per_sess.get(sid, []):
                if aid not in used:
                    res.append(aid); used.add(aid)
                if len(res)==20: break
        out[sid] = res
    return out

def _batch_predict_head(head, df_sorted, alpha=0.35, batch_sessions=120000):
    assert head in ("clicks","buys")
    model = ranker_clicks if head=="clicks" else ranker_buys
    preds_all = {}
    sessions = df_sorted["session"].drop_duplicates().values
    N = len(sessions)
    for start in range(0, N, batch_sessions):
        end = min(N, start+batch_sessions)
        sess_batch = set(sessions[start:end])
        g = df_sorted[df_sorted["session"].isin(sess_batch)].groupby("session", sort=False)
        scores_per_sess = defaultdict(list)
        fallback = {}
        for sid, df_sess in g:
            (aids, types, unique_aids, unique_buys,
             agg_click, agg_buys, agg_b2b,
             cand_clicks, cand_buys) = build_candidates(df_sess)
            if head=="clicks":
                X, keys = make_features_for_head(df_sess, "clicks", cand_clicks,
                                                 aids, types, unique_aids, unique_buys,
                                                 agg_click, agg_buys, agg_b2b)
                fallback[sid] = suggest_clicks(df_sess)
                rule_score = (X['rec_rule'] + X['w_main'] + 0.2*X['best_main']).values if not X.empty else None
            else:
                X, keys = make_features_for_head(df_sess, "buys", cand_buys,
                                                 aids, types, unique_aids, unique_buys,
                                                 agg_click, agg_buys, agg_b2b)
                fallback[sid] = suggest_buys(df_sess)
                rule_score = (X['rec_rule'] + X['w_main'] + 0.2*X['best_main']).values if not X.empty else None
            if X.empty:
                continue
            # Dask ranker: we can only call predict on dask collections;
            # for simplicity, we'll fall back to rule-only score here and blend=rule if model missing API.
            try:
                sc = model.predict(X.values)
                sc = np.asarray(sc).reshape(-1)
            except Exception:
                sc = np.zeros(len(keys), dtype=np.float32)
            if alpha is not None and rule_score is not None:
                sc = alpha * sc + (1.0 - alpha) * rule_score
            for aid, s in zip(keys, sc):
                scores_per_sess[sid].append((float(s), int(aid)))
        batch_pred = _rank_take20_with_fallback(scores_per_sess, fallback)
        preds_all.update(batch_pred)
    return preds_all

# Build predictions
test_sorted = test_df.sort_values(["session","ts"], kind="mergesort")
clicks_top20 = _batch_predict_head("clicks", test_sorted, alpha=0.35)
buys_top20   = _batch_predict_head("buys",   test_sorted, alpha=0.35)

def _to_submission_block(d, suffix):
    s = pd.Series({f"{sid}_{suffix}": " ".join(map(str, aids)) for sid, aids in d.items()})
    df = s.rename_axis("session_type").reset_index(name="labels")
    return df

clicks_pred_df = _to_submission_block(clicks_top20, "clicks")
orders_pred_df = _to_submission_block(buys_top20,   "orders")
carts_pred_df  = _to_submission_block(buys_top20,   "carts")

sub_rules = pd.read_csv("submission_rules.csv") if os.path.exists("submission_rules.csv") else None
sub_blend = pd.concat([clicks_pred_df, orders_pred_df, carts_pred_df], ignore_index=True)
sub_blend.to_csv("submission_blend.csv", index=False)

print("submission_blend.csv rows:", sub_blend.shape[0])

## 11. Sanity Checks

In [ ]:
df = pd.read_csv("submission_blend.csv")
ok = (df["labels"].str.split().map(len) == 20).mean()
print("20-per-line ratio:", f"{ok:.3f}")
df.head()